# Dominios raros y baja prevalencia

## Objetivo

Encontrar dominios consultados por pocos clientes y enriquecerlos con identidad DHCP.

## Entradas esperadas

- Ventana de tiempo.
- Minimo y maximo de clientes por dominio.

## Requisitos

- Acceso al area de trabajo de Microsoft Sentinel.
- Funciones KQL publicadas: `fn_Normalize_Windows_DHCP`, `fn_Normalize_Windows_DNS`, `fn_Correlate_DHCP_DNS`.
- Paquetes Python sugeridos: `msticpy`, `pandas`, `matplotlib`, `plotly`, `networkx` segun el notebook.

## Secciones

1. Prevalencia de dominios.
2. Hosts que consultaron dominios raros.
3. Tipos de consulta.
4. Evidencia para escalamiento.


In [ ]:
# Configuracion general - ajustar antes de ejecutar
workspace_id = "REEMPLAZAR_CON_WORKSPACE_ID"
tenant_id = "REEMPLAZAR_CON_TENANT_ID"

# Conexion sugerida con MSTICPy
# import msticpy as mp
# mp.init_notebook(namespace=globals())
# qry_prov = mp.QueryProvider("MSSentinel")
# qry_prov.connect(workspace=workspace_id, tenant_id=tenant_id)


In [ ]:
query_rare_domains = """
let Lookback = 14d;
let MinOrgClients = 1;
let MaxOrgClients = 3;
let dns = fn_Correlate_DHCP_DNS(Lookback)
| where IsReverseLookup == false
| where IsInternalName == false
| where isnotempty(QueryRootDomain);
let prevalence =
    dns
    | summarize OrgClientCount=dcount(ClientIp), OrgHosts=make_set(HostName, 20), FirstSeen=min(TimeGenerated), LastSeen=max(TimeGenerated) by QueryRootDomain
    | where OrgClientCount between (MinOrgClients .. MaxOrgClients);
dns
| join kind=inner prevalence on QueryRootDomain
| summarize Queries=count(), SampleQueries=make_set(QueryName, 20), QueryTypes=make_set(QueryType, 10), FirstSeen=min(TimeGenerated), LastSeen=max(TimeGenerated) by ClientIp, HostName, ClientMac, QueryRootDomain, OrgClientCount
| order by OrgClientCount asc, Queries desc
"""
# rare_domains_df = qry_prov.exec_query(query_rare_domains)
print(query_rare_domains)


## Resumen para incidente

Documentar aqui:

- Hallazgos principales.
- Entidades relevantes: IP, hostname, direccion MAC, dominio.
- Evidencia KQL usada.
- Recomendacion: cerrar, monitorear, escalar o contener.
